In [54]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

# 加载环境变量
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


# 初始化模型
# model = init_chat_model("groq:llama-3.3-70b-versatile", api_key=GROQ_API_KEY)
model = init_chat_model("groq:qwen/qwen3-32b", api_key=GROQ_API_KEY)

In [7]:
@tool
def calculator(operation: str, a: float, b: float) -> str:
    """执行数学计算"""
    ops = {
        "add": lambda x, y: x + y,
        "multiply": lambda x, y: x * y,
    }
    result = ops.get(operation, lambda x, y: 0)(a, b)
    return f"{a} {operation} {b} = {result}"

In [14]:
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
        checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": "long_conversation"}}

# 模拟多轮对话

for i in range(1, 11):
    response=agent.invoke(
        {"messages": [{"role": "user", "content": f"这是第 {i} 轮对话"}]},
        config=config
    )
    print("对话：",i)
    print("--------",response["messages"][i])

# 查看消息数量
response = agent.invoke(
    {"messages": [{"role": "user", "content": "总结一下"}]},
    config=config
)

print(f"\n总消息数: {len(response['messages'])}")

对话： 1
-------- content='你好，我很高兴开始我们的对话。你今天过得怎么样？还有什么我可以帮你做的吗？' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 51, 'total_tokens': 79, 'completion_time': 0.12325523, 'completion_tokens_details': None, 'prompt_time': 0.002971609, 'prompt_tokens_details': None, 'queue_time': 0.183870411, 'total_time': 0.126226839}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_e65acd3773', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d47c2-1598-7c72-935f-5e8dfef56727-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 51, 'output_tokens': 28, 'total_tokens': 79}
对话： 2
-------- content='这是第 2 轮对话' additional_kwargs={} response_metadata={} id='7d49dd84-32c2-4d47-8cf1-3fc455e93a02'
对话： 3
-------- content='看起来我们处于对话的早期阶段。你觉得聊点什么好呢？你想讨论一个特定话题，还是随便聊聊？' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 96, 't

In [22]:
response['messages']

[HumanMessage(content='我叫张三，是工程师', additional_kwargs={}, response_metadata={}, id='d25744b5-864f-453e-b931-94c7a5c280b6'),
 AIMessage(content='你好，张三！很高兴认识你。作为一名工程师，你一定在你的领域中非常出色。是什么让你对工程感兴趣的？你目前在从事什么样的项目？我很乐意与你聊聊你的工作和兴趣。', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 50, 'total_tokens': 111, 'completion_time': 0.341032853, 'completion_tokens_details': None, 'prompt_time': 0.006627301, 'prompt_tokens_details': None, 'queue_time': 0.096837994, 'total_time': 0.347660154}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d47cb-f202-7b91-a349-d2e5cb294aeb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 61, 'total_tokens': 111}),
 HumanMessage(content='我在北京工作', additional_kwargs={}, response_metadata={}, id='802399c4-c8d5-4c5d-9609-d678105aab5e'),
 A

In [23]:
response['messages'][i].response_metadata

{'token_usage': {'completion_tokens': 61,
  'prompt_tokens': 50,
  'total_tokens': 111,
  'completion_time': 0.341032853,
  'completion_tokens_details': None,
  'prompt_time': 0.006627301,
  'prompt_tokens_details': None,
  'queue_time': 0.096837994,
  'total_time': 0.347660154},
 'model_name': 'llama-3.3-70b-versatile',
 'system_fingerprint': 'fp_68f543a7cc',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

In [20]:
agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你是一个有帮助的助手。",
        checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.3-70b-versatile",
            max_tokens_before_summary=500  # 超过 500 tokens 就摘要
        )
    ]
)

config = {"configurable": {"thread_id": "with_summary"}}


conversations = [
    "我叫张三，是工程师",
    "我在北京工作",
    "我喜欢编程和阅读",
    "我最近在学习 AI",
    "请总结一下我的信息"
]

for msg in conversations:
    i=0
    print(f"\n用户: {msg}")
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    print(f"Agent: {response['messages'][-1].content[:100]}...")
    print(response['messages'][i].response_metadata)
    i+=1

print(f"\n消息数: {len(response['messages'])}")


/var/folders/zt/63tm27h17v75fy_47rqppsl40000gn/T/ipykernel_70346/3858938678.py:7: DeprecationWarning: max_tokens_before_summary is deprecated. Use trigger=('tokens', value) instead.
  SummarizationMiddleware(



用户: 我叫张三，是工程师
Agent: 你好，张三！很高兴认识你。作为一名工程师，你一定在你的领域中非常出色。是什么让你对工程感兴趣的？你目前在从事什么样的项目？我很乐意与你聊聊你的工作和兴趣。...
{}

用户: 我在北京工作
Agent: 北京是一座伟大的城市，拥有丰富的历史和文化。作为一名在北京工作的工程师，你一定经历了这座城市快速发展和创新带来的变化。北京的科技行业非常发达，你可能有机会参与到一些非常有趣和具有挑战性的项目中。

你...
{}

用户: 我喜欢编程和阅读
Agent: 编程和阅读是两个非常好的爱好。编程可以让你创造出新的东西，解决问题，实现你的想法。而阅读可以让你拓宽视野，获得新知识，放松身心。

你最喜欢编程哪种语言？你有最喜欢的编程项目或应用程序吗？你喜欢阅读什...
{}

用户: 我最近在学习 AI
Agent: 人工智能（AI）是一个非常令人兴奋的领域，近年来正在迅速发展。学习AI可以让你了解到机器学习、深度学习、自然语言处理等技术的原理和应用。

你对AI的哪些方面最感兴趣？是计算机视觉、自然语言处理，还是...
{}

用户: 请总结一下我的信息
Agent: 以下是您信息的总结：

* 您的名字是张三
* 您是一名工程师
* 您在北京工作
* 您喜欢编程和阅读
* 您最近在学习人工智能（AI）

如果您想添加或修改任何信息，请告诉我！...
{}

消息数: 10


In [25]:
response['messages'][i].response_metadata

{'token_usage': {'completion_tokens': 61,
  'prompt_tokens': 50,
  'total_tokens': 111,
  'completion_time': 0.341032853,
  'completion_tokens_details': None,
  'prompt_time': 0.006627301,
  'prompt_tokens_details': None,
  'queue_time': 0.096837994,
  'total_time': 0.347660154},
 'model_name': 'llama-3.3-70b-versatile',
 'system_fingerprint': 'fp_68f543a7cc',
 'service_tier': 'on_demand',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'groq'}

In [ ]:
from langchain_core.messages import trim_messages

# 模拟一个长对话历史
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content="我叫张三，是工程师"),
    AIMessage(content="你好，张三！很高兴认识你。作为一名工程师，你一定在你的领域中非常出色。是什么让你对工程感兴趣的？你目前在从事什么样的项目？我很乐意与你聊聊你的工作和兴趣。"),
    HumanMessage(content="我在北京工作"),
    AIMessage(content="北京是一座伟大的城市，拥有丰富的历史和文化。作为一名在北京工作的工程师，你一定经历了这座城市快速发展和创新带来的变化。"),
    HumanMessage(content="我喜欢编程和阅读"),
    AIMessage(content="编程和阅读是两个非常好的爱好。编程可以让你创造出新的东西，解决问题，实现你的想法。而阅读可以让你拓宽视野，获得新知识，放松身心。"),
    HumanMessage(content="我最近在学习 AI"),
    AIMessage(content="人工智能（AI）是一个非常令人兴奋的领域，近年来正在迅速发展。学习AI可以让你了解到机器学习、深度学习、自然语言处理等技术的原理和应用。"),
]

print(f"\n原始消息数: {len(messages)}")

def token_counter(messages):
    return sum(len(msg.content) for msg in messages)
    
trimmed = trim_messages(
    messages,
    max_tokens=70,  # 使用 token 数限制
    strategy="last",  # 保留最后的消息(first,last)
    # token_counter=len  
    token_counter=token_counter #按字符长度算
)

print(f"修剪后消息数: {len(trimmed)}")
print("\n保留的消息：")
for msg in trimmed:
    print(f"  {msg.__class__.__name__}: {msg.content}")



原始消息数: 8
修剪后消息数: 1

保留的消息：
  AIMessage: 人工智能（AI）是一个非常令人兴奋的领域，近年来正在迅速发展。学习AI可以让你了解到机器学习、深度学习、自然语言处理等技术的原理和应用。


In [52]:
from langchain_core.messages import trim_messages

# 模拟一个长对话历史
from langchain_core.messages import HumanMessage, AIMessage

messages = [
    HumanMessage(content="我叫张三，是工程师"),
    AIMessage(content="你好，张三！很高兴认识你。作为一名工程师，你一定在你的领域中非常出色。是什么让你对工程感兴趣的？你目前在从事什么样的项目？我很乐意与你聊聊你的工作和兴趣。"),
    HumanMessage(content="我在北京工作"),
    AIMessage(content="北京是一座伟大的城市，拥有丰富的历史和文化。作为一名在北京工作的工程师，你一定经历了这座城市快速发展和创新带来的变化。"),
    HumanMessage(content="我喜欢编程和阅读"),
    AIMessage(content="编程和阅读是两个非常好的爱好。编程可以让你创造出新的东西，解决问题，实现你的想法。而阅读可以让你拓宽视野，获得新知识，放松身心。"),
    HumanMessage(content="我最近在学习 AI"),
    AIMessage(content="人工智能（AI）是一个非常令人兴奋的领域，近年来正在迅速发展。学习AI可以让你了解到机器学习、深度学习、自然语言处理等技术的原理和应用。"),
]

print(f"\n原始消息数: {len(messages)}")

# token_counter=len
# 👉 所以它其实变成了：
# 按“消息条数”来算，而不是 token
# 📌 结果：最多保留 5 条消息

trimmed = trim_messages(
    messages,
    max_tokens=5,  # 或使用 token 数限制
    strategy="last",  # 保留最后的消息(first,last)
    token_counter=len  
)


print(f"修剪后消息数: {len(trimmed)}")
print("\n保留的消息：")
for msg in trimmed:
    print(f"  {msg.__class__.__name__}: {msg.content}")



原始消息数: 8
修剪后消息数: 5

保留的消息：
  AIMessage: 北京是一座伟大的城市，拥有丰富的历史和文化。作为一名在北京工作的工程师，你一定经历了这座城市快速发展和创新带来的变化。
  HumanMessage: 我喜欢编程和阅读
  AIMessage: 编程和阅读是两个非常好的爱好。编程可以让你创造出新的东西，解决问题，实现你的想法。而阅读可以让你拓宽视野，获得新知识，放松身心。
  HumanMessage: 我最近在学习 AI
  AIMessage: 人工智能（AI）是一个非常令人兴奋的领域，近年来正在迅速发展。学习AI可以让你了解到机器学习、深度学习、自然语言处理等技术的原理和应用。
